# Spatial comparison on the common Noah-MP grid

Three figure types, separately for SWE and snow depth:
1. Difference maps: iSnobal - Noah-MP, iSnobal - ASO, Noah-MP - ASO.
2. Snow-state maps: ASO, iSnobal, Noah-MP.
3. Elevation-bin boxplots of those same three differences.

Dates, in order: 2018-05-24, 2022-04-21, 2023-04-01, 2026-04-05.
ASO comes directly from the original GeoTIFFs, without additional QC.
File-declared NoData is excluded; finite negative values are NOT removed.
ASO is converted from meters to mm and area-averaged with the existing
workflow function and coverage threshold. Saved model rasters are already mm.
All figures/statistics use the same three-source finite mask within the
watershed for each date/variable. Each retained Noah cell counts equally,
including partially intersecting watershed cells; fractional area is used
for inclusion, not as a statistical weight. Snow-free zeros are retained.
Elevation uses Noah-MP HGT_M and the existing workflow bin definitions.
The All group includes Lower, Middle, and Upper; it is not an independent bin.

Box = Q1-Q3; black line = median; yellow diamond = mean; whiskers extend
to the most extreme observations within 1.5 IQR, and outliers are shown.
No notches are used: these are descriptive spatial distributions, without
an asserted independent-cell confidence interval for the median.
Each date has its own colorbar, shared across that date's three maps.
Boxplot y limits are independent by date, include zero and all outliers,
and have 6% padding. By default no values are clipped. No smoothing.

Run in the East River HPC environment with its geospatial dependencies.
Notebook: run all cells in order; then rerun individual figure cells to style.
Script: python Spatial_comparison_raw_ASO.py

In [ ]:
from pathlib import Path
import sys
import json
import warnings
import numpy as np
import pandas as pd
import rasterio
from pyproj import CRS
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.colors import BoundaryNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from PIL import Image

OUTPUT_DIR = Path('/bsuhome/zihaozhang/scratch/iSnobal/East_River_Workflow_Code_v0.8.2/output/output1')
REPO_CANDIDATES = [
    Path('/bsuhome/zihaozhang/scratch/iSnobal/East_River_Workflow_Code_v0.8.2'),
    Path('/bsuscratch/zihaozhang/iSnobal/East_River_Workflow_Code_v0.8.2'),
]
DATES = ['20180524', '20220421', '20230401', '20260405']
CASE_NAMES = ['Low snow / late season', 'Typical / close agreement',
              'High snow', 'Strong model contrast']
VARIABLES = ['SWE', 'Snow depth']  # Use ['SWE'] to generate only main-text SWE figures.
FONT_NAME = 'Nimbus Sans'
EXPORT_DPI = 600
SHOW_FIGURES = True
FIGSIZE_MAP = (18, 22)
FIGSIZE_BOX = (18, 12)
# Optional increasing map boundaries in mm; None = automatic for each date.
# A variable can also use a date dictionary, e.g. {'20180524': [0, 50, 100, 150]}.
STATE_BOUNDS = {'SWE': None, 'Snow depth': None}
DIFF_BOUNDS = {'SWE': None, 'Snow depth': None}
BOX_Y_PADDING = 0.06  # Fraction of each date's full range; zero is included.
FIG_DIR = OUTPUT_DIR / 'figures/publication/spatial_raw_aso'
TABLE_DIR = OUTPUT_DIR / 'tables/publication/spatial_raw_aso'
PAIR_NAMES = ['iSnobal - Noah-MP', 'iSnobal - ASO', 'Noah-MP - ASO']
PAIR_COLORS = ['0.55', '#1F77B4', '#D62728']
BANDS = ['Lower', 'Middle', 'Upper', 'All']
BAND_LABELS = ['Lower\n<2896 m', 'Middle\n2896–<3353 m', 'Upper\n≥3353 m', 'All']
WHISKER_IQR = 1.5
SHOW_FLIERS = True
RASTER_ATOL_MM = 0.002  # Saved float32 raster/subtraction tolerance, not rounding.

In [ ]:
def slug(variable):
    return variable.lower().replace(' ', '_')


def configure_style():
    if FONT_NAME not in {f.name for f in fm.fontManager.ttflist}:
        raise RuntimeError(f'{FONT_NAME} is required; install it before publication export.')
    print('Font:', fm.findfont(fm.FontProperties(family=FONT_NAME), fallback_to_default=False))
    mpl.rcParams.update({
        'font.family': 'sans-serif', 'font.sans-serif': [FONT_NAME],
        'font.size': 18, 'axes.titlesize': 20, 'font.weight': 'normal',
        'axes.titleweight': 'normal', 'axes.labelweight': 'normal',
        'axes.linewidth': 1.0, 'pdf.fonttype': 42, 'ps.fonttype': 42,
    })


def load_context():
    repo = next((p for p in REPO_CANDIDATES if (p/'src/east_river_workflow').is_dir()), None)
    if repo is None:
        raise FileNotFoundError('Cannot find East River workflow source tree. Edit REPO_CANDIDATES.')
    sys.path.insert(0, str(repo/'src'))
    from east_river_workflow.config import load_config
    from east_river_workflow.data_access import read_noah_grid, read_aso_raster
    from east_river_workflow.grids import load_weights, select_watershed, regrid_area_average
    from east_river_workflow.aso import elevation_band_mask
    cfg = load_config(repo/'config/east_river_config.yaml')
    if Path(cfg.output_dir).resolve() != OUTPUT_DIR.resolve():
        warnings.warn(f'Config output path is {cfg.output_dir}; explicit OUTPUT_DIR={OUTPUT_DIR} takes precedence.')
    grid, dem, _, _, _ = read_noah_grid(cfg.data['paths']['noahmp_geo_file'], cfg.section('noahmp'))
    weights = load_weights(OUTPUT_DIR/'cache/east_river_fraction_noahmp.npz', grid)
    if np.any(np.isfinite(weights) & ((weights < 0) | (weights > 1))):
        raise ValueError('Watershed fractional weights lie outside [0, 1].')
    domain = np.isfinite(weights) & (weights > 0)
    if not domain.any():
        raise ValueError('No valid watershed cells.')
    if not np.isfinite(dem[domain]).all():
        raise ValueError('Noah DEM has missing watershed elevations; resolve before binning.')
    wcfg = cfg.section('watershed')
    boundary = select_watershed(cfg.data['paths']['watershed_shapefile'],
                                wcfg['name_field'], wcfg['name_contains']).to_crs(grid.crs)
    if not grid.crs.is_projected:
        raise ValueError('Expected a projected Noah grid for consistent spatial maps.')
    if not np.allclose([grid.transform.b, grid.transform.d], 0):
        raise ValueError('Rotated raster affine is unsupported; do not silently display it as north-up.')
    return dict(cfg=cfg, grid=grid, dem=dem, weights=weights, domain=domain,
                boundary=boundary, read_raw=read_aso_raster, regrid=regrid_area_average,
                band_mask=elevation_band_mask, repo=repo)


def raster_file(date, variable, key):
    return OUTPUT_DIR/'regridded'/f'{date}_{slug(variable)}_{key}.tif'


def read_aligned(path, grid):
    with rasterio.open(path) as src:
        if src.count != 1 or (src.height, src.width) != grid.shape:
            raise ValueError(f'Raster shape/band conflict: {path}')
        if src.crs is None or CRS(src.crs) != grid.crs:
            raise ValueError(f'Raster CRS conflict: {path}')
        if not np.allclose(tuple(src.transform), tuple(grid.transform), rtol=0, atol=1e-8):
            raise ValueError(f'Raster transform conflict: {path}')
        return src.read(1, masked=True).astype(float).filled(np.nan)


def summarize_difference(reference, model):
    reference, model = np.asarray(reference, float), np.asarray(model, float)
    if reference.shape != model.shape:
        raise ValueError('Paired arrays have different shapes.')
    ok = np.isfinite(reference) & np.isfinite(model)
    r, m = reference[ok], model[ok]
    if not len(r):
        return dict(n=0, reference_mean_mm=np.nan, mean_mm=np.nan,
                    median_mm=np.nan, rmse_mm=np.nan, RMB_pct=np.nan,
                    RRMSE_pct=np.nan, q25_mm=np.nan, q75_mm=np.nan)
    diff = m-r
    ref_mean = float(r.mean())
    mb, rmse = float(diff.mean()), float(np.sqrt(np.mean(diff**2)))
    return dict(n=len(r), reference_mean_mm=ref_mean, mean_mm=mb,
                median_mm=float(np.median(diff)), rmse_mm=rmse,
                RMB_pct=100*mb/ref_mean if ref_mean != 0 else np.nan,
                RRMSE_pct=100*rmse/ref_mean if ref_mean != 0 else np.nan,
                q25_mm=float(np.percentile(diff,25)), q75_mm=float(np.percentile(diff,75)))


def load_acquisition(ctx, date, variable):
    grid, cfg, domain = ctx['grid'], ctx['cfg'], ctx['domain']
    i_path = raster_file(date,variable,'isnobal_on_noah')
    n_path = raster_file(date,variable,'noah_native')
    isnobal, noah = read_aligned(i_path,grid), read_aligned(n_path,grid)
    pattern = cfg.section('aso')['swe_pattern' if variable=='SWE' else 'snow_depth_pattern']
    raw_path = Path(cfg.data['paths']['aso_directory']).expanduser()/pattern.format(date=date)
    raw, raw_grid, metadata = ctx['read_raw'](raw_path)
    coverage_limit = float(cfg.section('aso')['minimum_valid_coverage'])
    aso, coverage = ctx['regrid'](raw, raw_grid, grid, minimum_valid_coverage=coverage_limit)
    aso = np.where(domain, aso, np.nan).astype(float)
    common = domain & np.isfinite(aso) & np.isfinite(isnobal) & np.isfinite(noah)
    if not common.any():
        raise ValueError(f'No shared ASO/model support on {date}, {variable}.')
    raw_common = [aso, isnobal, noah]
    fields = [np.where(common,a,np.nan) for a in raw_common]
    differences = [fields[1]-fields[2], fields[1]-fields[0], fields[2]-fields[0]]
    audit = dict(date=pd.Timestamp(date).date().isoformat(), variable=variable,
                 raw_aso_path=str(raw_path), isnobal_path=str(i_path), noah_path=str(n_path),
                 minimum_valid_coverage=coverage_limit, common_cells=int(common.sum()),
                 aso_negative_common_cells=int((aso[common]<0).sum()),
                 aso_common_mean_mm=float(aso[common].mean()),
                 model_pair_cells=int((domain & np.isfinite(isnobal) & np.isfinite(noah)).sum()),
                 asoi_pair_cells=int((domain & np.isfinite(aso) & np.isfinite(isnobal)).sum()),
                 ason_pair_cells=int((domain & np.isfinite(aso) & np.isfinite(noah)).sum()),
                 saved_aso_comparison='not available', saved_aso_max_abs_difference_mm=np.nan,
                 saved_aso_support_mismatch_cells=np.nan)
    saved = raster_file(date,variable,'aso_on_noah')
    if saved.exists():
        old = read_aligned(saved,grid)
        old_mask, new_mask = domain & np.isfinite(old), domain & np.isfinite(aso)
        both = old_mask & new_mask
        max_diff = float(np.max(np.abs(old[both]-aso[both]))) if both.any() else np.nan
        support_mismatch = int(np.count_nonzero(old_mask != new_mask))
        matches = support_mismatch==0 and both.any() and max_diff<=RASTER_ATOL_MM
        audit.update(saved_aso_comparison='matches raw regrid within tolerance' if matches else 'DIFFERS from raw regrid',
                     saved_aso_max_abs_difference_mm=max_diff,
                     saved_aso_support_mismatch_cells=support_mismatch)
        if not matches:
            warnings.warn(f'{date} {variable}: saved ASO differs from the new raw-ASO regrid. '
                          'Raw ASO is used as requested; see acquisition_audit.csv.')
    return dict(date=date, variable=variable, fields=fields, differences=differences,
                common=common, audit=audit, aso_coverage=coverage)


def build_tables(ctx, products):
    summaries, cells = [], []
    for (date,variable), p in products.items():
        a,i,n = p['fields']
        for band in BANDS:
            mask = p['common'] & ctx['band_mask'](ctx['dem'],band)
            for name,reference,model in zip(PAIR_NAMES,[n,a,a],[i,i,n]):
                summaries.append(dict(date=pd.Timestamp(date).date().isoformat(), variable=variable,
                                      elevation_band=band, comparison=name,
                                      **summarize_difference(reference[mask],model[mask])))
        rr,cc = np.where(p['common'])
        cells.append(pd.DataFrame(dict(date=pd.Timestamp(date).date().isoformat(), variable=variable,
            row=rr, col=cc, elevation_m=ctx['dem'][rr,cc],
            watershed_fraction=ctx['weights'][rr,cc], aso_mm=a[rr,cc], isnobal_mm=i[rr,cc],
            noahmp_mm=n[rr,cc], isnobal_minus_noahmp_mm=(i-n)[rr,cc],
            isnobal_minus_aso_mm=(i-a)[rr,cc], noahmp_minus_aso_mm=(n-a)[rr,cc])))
    return pd.DataFrame(summaries), pd.concat(cells,ignore_index=True)


def compare_table6(summary):
    path = OUTPUT_DIR/'tables/table6_aso_validation.csv'
    if not path.exists():
        return pd.DataFrame([dict(status='Table 6 unavailable; raw-ASO figures still valid')])
    previous = pd.read_csv(path)
    required = {'date','variable','product','elevation_band','mean_mm','rmse_mm'}
    if not required.issubset(previous):
        raise ValueError(f'Table 6 schema conflict: missing {sorted(required-set(previous))}')
    previous['date'] = pd.to_datetime(previous['date'].astype(str)).dt.strftime('%Y-%m-%d')
    records=[]
    for row in summary.itertuples(index=False):
        product={'iSnobal - ASO':'iSnobal-ASO on Noah-MP grid',
                 'Noah-MP - ASO':'Noah-MP-ASO on Noah-MP grid'}.get(row.comparison)
        if product is None: continue
        old=previous.loc[(previous.date==row.date)&(previous.variable==row.variable)
                         &(previous.elevation_band==row.elevation_band)&(previous['product']==product)]
        result=dict(date=row.date, variable=row.variable, elevation_band=row.elevation_band,
                    comparison=row.comparison, raw_common_n=row.n, raw_common_MB_mm=row.mean_mm,
                    raw_common_RMSE_mm=row.rmse_mm)
        if len(old)!=1:
            result.update(status=f'Expected one matching Table 6 row; found {len(old)}')
        else:
            entry=old.iloc[0]
            match=(np.isclose(row.mean_mm,entry.mean_mm,atol=RASTER_ATOL_MM,rtol=0,equal_nan=True)
                   and np.isclose(row.rmse_mm,entry.rmse_mm,atol=RASTER_ATOL_MM,rtol=0,equal_nan=True))
            result.update(old_MB_mm=entry.mean_mm, old_RMSE_mm=entry.rmse_mm)
            if 'cell_count' in entry:
                result['old_n']=entry.cell_count
                match=match and row.n==entry.cell_count
            result['status']='matches' if match else 'DIFFERS: raw ASO and/or common support differs'
        records.append(result)
    audit=pd.DataFrame(records)
    if (audit.status!='matches').any():
        warnings.warn('Some raw-ASO/common-support results differ from Table 6 or lack a reference row. '
                      'See table6_comparison_audit.csv; no previous table values are substituted.')
    return audit


def export_audits(ctx,products,summary,cell_values):
    TABLE_DIR.mkdir(parents=True,exist_ok=True)
    summary.to_csv(TABLE_DIR/'spatial_metrics_by_elevation.csv',index=False)
    cell_values.to_csv(TABLE_DIR/'spatial_difference_cell_values.csv',index=False)
    pd.DataFrame([p['audit'] for p in products.values()]).to_csv(TABLE_DIR/'acquisition_audit.csv',index=False)
    compare_table6(summary).to_csv(TABLE_DIR/'table6_comparison_audit.csv',index=False)
    provenance=dict(aso_source='original ASO rasters; no additional QC', units='mm',
        source_tree=str(ctx['repo']), output_dir=str(OUTPUT_DIR), dates=DATES, variables=VARIABLES,
        support='same finite ASO/iSnobal/Noah cells per date and variable; watershed fraction > 0',
        weighting='one common-grid cell = one equally weighted observation',
        elevation_bins={'Lower':'<2896 m','Middle':'2896 <= z <3353 m','Upper':'>=3353 m','All':'all retained cells'},
        differences=PAIR_NAMES, box_whiskers='1.5 IQR; show all outliers; no notches',
        raw_aso_note='Raw recomputation may change statistics from the screenshot; selected dates are unchanged.')
    (TABLE_DIR/'provenance.json').write_text(json.dumps(provenance,indent=2),encoding='utf-8')

In [ ]:
def nice_max(value):
    if not np.isfinite(value) or value<=0: return 1.0
    magnitude=10.0**np.floor(np.log10(value))
    for factor in [1,1.5,2,2.5,3,4,5,6,8,10]:
        if factor*magnitude>=value: return factor*magnitude
    return 10*magnitude


def map_scale(products,variable,date,difference=False):
    key='differences' if difference else 'fields'
    vals=np.concatenate([a[np.isfinite(a)] for a in products[date,variable][key]])
    prescribed=(DIFF_BOUNDS if difference else STATE_BOUNDS)[variable]
    if isinstance(prescribed,dict): prescribed=prescribed.get(date)
    if prescribed is None:
        if difference:
            bound=nice_max(np.max(np.abs(vals)))
            bounds=np.linspace(-bound,bound,9)
        else:
            lo=-nice_max(abs(vals.min())) if vals.min()<0 else 0.0
            bounds=np.linspace(lo,nice_max(vals.max()),9)
    else:
        bounds=np.asarray(prescribed,dtype=float)
        if bounds.ndim!=1 or len(bounds)<3 or not np.isfinite(bounds).all() or not (np.diff(bounds)>0).all():
            raise ValueError('Map bounds must be a finite, strictly increasing list.')
        if difference and (not bounds[0]<0<bounds[-1] or not np.allclose(bounds,-bounds[::-1])):
            raise ValueError('Difference bounds must be symmetric about zero.')
    cmap = plt.get_cmap('RdBu' if difference else 'YlGnBu').resampled(len(bounds)-1)
    cmap=cmap.copy(); cmap.set_bad('white')
    cmap.set_under('#67001f' if difference else '#e5e5e5')  # Negative: red
    cmap.set_over('#053061' if difference else '#081d58')   # Positive: blue
    under,over=int((vals<bounds[0]).sum()),int((vals>bounds[-1]).sum())
    extend='both' if under and over else ('min' if under else ('max' if over else 'neither'))
    return bounds,cmap,BoundaryNorm(bounds,cmap.N),extend,dict(below_scale=under,above_scale=over)


def save_figure(fig,stem):
    FIG_DIR.mkdir(parents=True,exist_ok=True)
    fig.savefig(FIG_DIR/(stem+'.pdf'),bbox_inches='tight',pad_inches=.05,facecolor='white')
    png=FIG_DIR/(stem+'.png')
    fig.savefig(png,dpi=EXPORT_DPI,bbox_inches='tight',pad_inches=.05,facecolor='white')
    with Image.open(png) as im:
        if im.mode!='RGB':
            rgb=im.convert('RGB')
        else: rgb=None
    if rgb is not None:
        rgb.save(png,dpi=(EXPORT_DPI,EXPORT_DPI)); rgb.close()
    print('Saved:',FIG_DIR/(stem+'.pdf'))
    print('Saved:',png)
    if SHOW_FIGURES: plt.show()
    plt.close(fig)


def plot_maps(ctx,products,variable,difference=False):
    grid=ctx['grid']
    key='differences' if difference else 'fields'
    titles=PAIR_NAMES if difference else ['ASO','iSnobal','Noah-MP']
    fig,axes=plt.subplots(len(DATES),3,figsize=FIGSIZE_MAP,squeeze=False)
    row_scales=[]
    # Tight common extent around every included watershed cell, including boundary cells.
    rr,cc=np.where(ctx['domain'])
    x0,y0=grid.transform*(cc.min(),rr.min())
    x1,y1=grid.transform*(cc.max()+1,rr.max()+1)
    xlo,xhi=sorted([x0,x1]); ylo,yhi=sorted([y0,y1])
    padx,pady=(xhi-xlo)*.035,(yhi-ylo)*.035
    for row,date in enumerate(DATES):
        bounds,cmap,norm,extend,scale_audit=map_scale(products,variable,date,difference)
        for col,title in enumerate(titles):
            ax=axes[row,col]
            image=ax.imshow(products[date,variable][key][col],extent=grid.extent,
                            origin='upper' if grid.transform.e<0 else 'lower',
                            cmap=cmap,norm=norm,interpolation='nearest')
            ctx['boundary'].boundary.plot(ax=ax,color='black',linewidth=.8)
            ax.set_xlim(xlo-padx,xhi+padx); ax.set_ylim(ylo-pady,yhi+pady)
            ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values(): spine.set_visible(False)
            if row==0: ax.set_title(title,pad=16)
            if col==0:
                ax.set_ylabel(f'{pd.Timestamp(date):%d %b %Y}\n{CASE_NAMES[row]}',labelpad=14)
            ax.text(.01,.98,f'({chr(97+row*3+col)})',transform=ax.transAxes,
                    ha='left',va='top',fontsize=18,zorder=5,
                    bbox=dict(facecolor='white',edgecolor='none',alpha=.85,pad=1.5))
        row_scales.append((image,bounds,extend,dict(date=date,bounds_mm=bounds.tolist(),**scale_audit)))
    fig.subplots_adjust(left=.13,right=.84,top=.95,bottom=.035,wspace=.03,hspace=.08)
    # Apply the equal-aspect map layout before matching each colorbar to its row.
    fig.canvas.draw()
    for row,(image,bounds,extend,_) in enumerate(row_scales):
        pos=axes[row,2].get_position()
        cax=fig.add_axes([.865,pos.y0,.016,pos.height])
        cb=fig.colorbar(image,cax=cax,orientation='vertical',ticks=bounds,extend=extend)
        cb.ax.tick_params(labelsize=14)
        cb.set_label(f'{variable} difference (mm)' if difference else f'{variable} (mm)',fontsize=16)
    kind='differences' if difference else 'states'
    TABLE_DIR.mkdir(parents=True,exist_ok=True)
    (TABLE_DIR/f'{slug(variable)}_{kind}_color_scale.json').write_text(
        json.dumps(dict(scale_scope='per date; shared across three columns',
                        dates=[entry[3] for entry in row_scales]),indent=2),encoding='utf-8')
    save_figure(fig,f'raw_aso_{slug(variable)}_{kind}_4dates')


def box_y_limits(product):
    values=np.concatenate([a[np.isfinite(a)] for a in product['differences']])
    lo=min(0.0,float(values.min())); hi=max(0.0,float(values.max()))
    padding=(hi-lo)*BOX_Y_PADDING if hi>lo else 1.0
    return lo-padding,hi+padding


def plot_elevation_boxes(ctx,products,variable):
    fig,axes=plt.subplots(2,2,figsize=FIGSIZE_BOX,squeeze=False,sharey=False)
    axis_audit=[]
    offsets=[-.24,0,.24]
    for idx,(date,ax) in enumerate(zip(DATES,axes.flat)):
        product=products[date,variable]
        for comparison,offset,color in zip(range(3),offsets,PAIR_COLORS):
            for b,band in enumerate(BANDS):
                mask=product['common'] & ctx['band_mask'](ctx['dem'],band)
                values=product['differences'][comparison][mask]
                if not values.size: continue  # Empty bins remain blank; summary records n=0.
                bp=ax.boxplot([values],positions=[b+offset],widths=.21,patch_artist=True,
                    notch=False,whis=WHISKER_IQR,showfliers=SHOW_FLIERS,manage_ticks=False,
                    medianprops=dict(color='black',linewidth=2),
                    whiskerprops=dict(color='black',linewidth=1.3),
                    capprops=dict(color='black',linewidth=1.3),
                    boxprops=dict(edgecolor='black',linewidth=1.3),
                    flierprops=dict(marker='o',markersize=3,markerfacecolor=color,
                                    markeredgecolor='black',markeredgewidth=.35,alpha=.55))
                for box in bp['boxes']: box.set_facecolor(color); box.set_alpha(.74)
                ax.scatter(b+offset,float(values.mean()),marker='D',s=45,
                           facecolor='yellow',edgecolor='black',linewidth=.8,zorder=5)
        ax.axhline(0,color='0.25',linewidth=1,linestyle='--',zorder=1)
        ax.set_facecolor('0.94'); ax.set_axisbelow(True)
        ax.yaxis.grid(True,linestyle='--',linewidth=.8,alpha=.45); ax.xaxis.grid(False)
        ylim=box_y_limits(product)
        ax.set_xlim(-.6,3.6); ax.set_ylim(*ylim)
        ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=6,steps=[1,2,2.5,5,10]))
        axis_audit.append(dict(date=date,ymin_mm=ylim[0],ymax_mm=ylim[1]))
        ax.set_xticks(range(4)); ax.set_xticklabels(BAND_LABELS,fontsize=14)
        ax.set_title(f'{pd.Timestamp(date):%d %b %Y} | {CASE_NAMES[idx]}',fontsize=18,pad=10)
        ax.text(.02,.97,f'({chr(97+idx)})',transform=ax.transAxes,ha='left',va='top')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        ax.tick_params(width=1,length=5)
        if idx%2==0: ax.set_ylabel(f'{variable} difference (mm)')
    handles=[Patch(facecolor=c,edgecolor='black',alpha=.74,label=n) for c,n in zip(PAIR_COLORS,PAIR_NAMES)]
    handles.append(Line2D([],[],linestyle='none',marker='D',color='yellow',markeredgecolor='black',label='Mean'))
    # Boxed legend above panels avoids covering outliers while retaining source identity.
    fig.legend(handles=handles,loc='upper center',bbox_to_anchor=(.54,.995),ncol=4,
               fontsize=15,frameon=True,facecolor='white',edgecolor='0.4',fancybox=False,framealpha=.9)
    fig.subplots_adjust(left=.09,right=.985,top=.90,bottom=.08,wspace=.22,hspace=.30)
    TABLE_DIR.mkdir(parents=True,exist_ok=True)
    (TABLE_DIR/f'{slug(variable)}_box_y_limits.json').write_text(
        json.dumps(dict(scope='independent per date; zero and all outliers included',
                        padding_fraction=BOX_Y_PADDING,dates=axis_audit),indent=2),encoding='utf-8')
    save_figure(fig,f'raw_aso_{slug(variable)}_elevation_difference_boxes_4dates')

In [ ]:
if __name__ == '__main__':
    configure_style()
    context=load_context()
    products={(d,v):load_acquisition(context,d,v) for v in VARIABLES for d in DATES}
    summary,cell_values=build_tables(context,products)
    export_audits(context,products,summary,cell_values)
    print(summary.to_string(index=False))

## 1 Spatial difference maps
Columns: iSnobal - Noah-MP, iSnobal - ASO, Noah-MP - ASO.
Positive means the first named dataset contains more snow.
Each date gets its own symmetric scale shared across its three comparison maps.

In [ ]:
if __name__ == '__main__':
    for variable in VARIABLES:
        plot_maps(context,products,variable,difference=True)

## 2 Spatial snow-state maps
Columns: ASO, iSnobal, Noah-MP. Identical grid, mask, and units in every column.
Each date has one colorbar shared by its three source maps.
Read the individual colorbars when comparing magnitudes across dates.

In [ ]:
if __name__ == '__main__':
    for variable in VARIABLES:
        plot_maps(context,products,variable,difference=False)

## 3 Elevation-bin difference boxplots
Each date is a panel; each elevation group contains the three comparisons.
Each date has independent y limits including zero and all outliers, with 6% padding.
The limits can be asymmetric. Read each panel's y ticks when comparing dates.
Empty bins are blank, not zero.
To change only styling, rerun this plotting cell after editing the functions.

In [ ]:
if __name__ == '__main__':
    for variable in VARIABLES:
        plot_elevation_boxes(context,products,variable)

## Outputs and interpretation
Six figures (three SWE, three snow depth), each in 600-dpi RGB PNG and PDF.
All files are placed in output1/figures/publication/spatial_raw_aso.
CSVs and provenance are placed in output1/tables/publication/spatial_raw_aso.
The acquisition audit compares raw-regridded ASO with the old saved ASO.
The Table 6 audit reports, rather than conceals, changed values/support.
Prior numbers such as the screenshot's ASO means are not hard-coded.
This notebook reads original inputs and existing products; it never overwrites
the original ASO rasters, model rasters, or previous workflow validation tables.